# 12.9 音訊怎麼進入既有模型？


圖片用佔位標記展開成多條視覺特徵，聲音也能沿用同一接口：音訊標記被替換成11條時間框特徵，接頭把它們轉成文字模型的寬度。回答文字在後面，只有回答位置計算猜錯代價。本節檢查這條路徑確實能傳遞訓練信號。

前置是[12.8時間特徵](https://birdhackor.github.io/tiny-perceptron-vlm/12.8.html)、[10.7展開與標籤錯位](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)和[11.3梯度檢查](https://birdhackor.github.io/tiny-perceptron-vlm/11.3.html)。音訊標記編號來自ByteTokenizer，不靠輸入字串中寫「音訊」兩字。目標high是人工設定的示範標籤，表示在我們的玩具規則裡440Hz超過300Hz，歸為高音。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.data import ByteTokenizer
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, tone

torch.manual_seed(0)
tok = ByteTokenizer()
model = MultiModalLM(TinyLM(ModelConfig(width=8)))
prefix = [tok.bos_id, tok.user_id, tok.audio_id, tok.eos_id, tok.assistant_id]
answer = tok.encode("high") + [tok.eos_id]
ids = torch.tensor(prefix + answer)
labels = torch.tensor([-100] * len(prefix) + answer)
out = model(ids, labels, waveform=tone(440))
loss = masked_loss(out["logits"], out["labels"])
loss.backward()
print("分數形狀", tuple(out["logits"].shape))
print("聲音接頭收到梯度", model.audio_projector.weight.grad.norm().item() > 0)

輸入包含錄音、角色與答案編號。prefix長度5，answer有high四個位元組加結束，共5格。原序列10格，音訊標記從1格變11格，增加10格，再作一次下一位置對齊去1格，因此分數形狀(1,19,264)。前兩軸是批次與展開後輸入長度，最後264是文字候選數。

`masked_loss`忽略前綴與音訊位置，只對五個答案目標計算；`backward`將其影響傳回聲音接頭，第二行通常True。這裡隨機模型還沒有訓練更新，所以high不是它預測正確的結果。訓練高低音任務需要更多頻率、平衡標籤與保留測試；也要先驗證編碼器特徵能提供相關差異。

相同接口讓角色、答案結束與文字loss規則繼續沿用，不需要為聲音重新寫一套對話邏輯。

音訊標記展開11格後，原來接在它後面的用戶結束、助手角色與high編號都往後10格。標籤裡的忽略區必須一起伸長，才能讓助手位置預測h、h預測i、i預測g、g預測h、最後h預測結束。這裡有五次有效目標，聲音11框本身沒有對應要寫出的文字標籤。這樣安排允許模型依據聲音決定回答，卻不要求它把每個時間框都硬翻成一個字。若直接把原標籤長度10套進19格輸出，既可能形狀報錯，也可能錯配監督；先核對展開長度與有效目標，才是接口調試的起點。

[正式音高問答](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/audio.json)已把這條路徑真正訓練起來：直接SFT文字底座接已訓練的音訊編碼器，全部開放更新300次，每次四段，累計5,409個有效回答目標。八個訓練頻率各有振幅0.25、時長0.1秒，以及振幅0.5、時長0.12秒兩種版本，共16段；驗證四個新頻率共8段，測試七個新頻率共14段。同一頻率的兩種版本留在同側，波形直接以每秒16,000點合成，沒有把同一頻率換個音量就稱為新頻率測試。

同一批訓練探針的代價從3.891降到約0.000045，驗證8/8，測試11/14，且14題都正常以EOS結束。三個錯題在邊界附近：290Hz較短、較小振幅的版本答high，300Hz兩種版本也答high；本規則把300Hz歸low。這不等於它學到精確的300Hz分界，下面還要換聲音檢查。這裡的11/14與12.8編碼器分類頭的11/14雖恰巧同分，錯題卻不同，前者是更新後生成文字的問答模型，後者是分類頭，不能沿用分類頭的校準信心作問答可靠度。

練習只把目標high改low，保持錄音440不變，先說明這是錯誤監督，因為違反本例300Hz邊界。程式仍可能收到梯度，這提醒我們路徑正常不代表資料標籤可靠。
